In [1]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue 

In [2]:
class Empty(Exception):
    pass

In [3]:
class DataProcessor:
    def __init__(self):
        self.queue = ArrayQueue()
        self.stack = ArrayStack()
        self.lista = []
        self.redo_stack = ArrayStack()

    def add(self, registro):
        if not isinstance(registro, tuple) or len(registro) != 3:
            raise ValueError('El registro debe ser una tupla de tres componentes: (sensor, variable, value).')
        if not isinstance(registro[2], (int, float)):
            raise ValueError('El value debe ser numérico (int o float)')
        self.queue.enqueue(registro)

    def process_next(self):
        if self.queue.is_empty():
            raise Empty('La cola de registros pendientes está vacía')
        
        registro = self.queue.dequeue()
        sensor = registro[0]
        variable = registro[1]

        posicion = -1

        for i in range(len(self.lista)):
            if self.lista[i][0] == sensor and self.lista[i][1] == variable:
                posicion = i
                break

        if posicion == -1:
            self.lista.append(registro)
            self.stack.push(('CREATE', len(self.lista) - 1, None))
        else:
            self.stack.push(('UPDATE', posicion, self.lista[posicion]))
            self.lista[posicion] = registro

        while not self.redo_stack.is_empty():
            self.redo_stack.pop()

        return registro

    def undo(self):
        if self.stack.is_empty():
            raise Empty('No hay cambios en el historial para deshacer.')
        tipo, posicion, anterior = self.stack.pop()
        aplicado = self.lista[posicion]
        if tipo == 'CREATE':
            self.lista.pop(posicion)
        else:
            self.lista[posicion] = anterior
        self.redo_stack.push(aplicado)

    def redo(self):
        if self.redo_stack.is_empty():
            raise Empty('No hay cambios para rehacer (redo).')

        registro = self.redo_stack.pop()
        sensor = registro[0]
        variable = registro[1]

        posicion = -1
        
        for i in range(len(self.lista)):
            if self.lista[i][0] == sensor and self.lista[i][1] == variable:
                posicion = i
                break
        
        if posicion == -1:
            self.lista.append(registro)
            self.stack.push(('CREATE', len(self.lista) - 1, None))
        else:
            self.stack.push(('UPDATE', posicion, self.lista[posicion]))
            self.lista[posicion] = registro

        return registro

    def pending(self):
        return len(self.queue)

    def current_value(self, sensor, variable):
        for i in range(len(self.lista)):
            if self.lista[i][0] == sensor and self.lista[i][1] == variable:
                return self.lista[i][2]
        raise KeyError((sensor, variable))        


##### Pruebas

In [8]:
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)

print("Registros de ejemplo:", A, B, C)

Registros de ejemplo: ('S01', 'temperature', 20) ('S01', 'temperature', 25) ('S01', 'humidity', 60)


In [17]:
#Prueba 1 pending() sobre un procesador vacío
p = DataProcessor()
print("1. pending() sobre procesador vacío:", p.pending())


1. pending() sobre procesador vacío: 0


In [16]:
#Prueba 2 agregar un registro
p = DataProcessor()
p.add(A)
print("2. después de add(A) -> pending():", p.pending())
print("   el estado sigue vacío (add no procesa):", p.lista)


2. después de add(A) -> pending(): 1
   el estado sigue vacío (add no procesa): []


In [15]:
#Prueba 3 agregar varios registros
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
print('3. después de add(A), add(B), add(C) -> pending():', p.pending())


3. después de add(A), add(B), add(C) -> pending(): 3


In [14]:
#Prueba 4 verificar procesamiento FIFO
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
orden = [p.process_next(), p.process_next(), p.process_next()]
print("4. orden de procesamiento:", orden)



4. orden de procesamiento: [('S01', 'temperature', 20), ('S01', 'temperature', 25), ('S01', 'humidity', 60)]


In [13]:
#Prueba 5 procesar un registro
p = DataProcessor()
p.add(A)
devuelto = p.process_next()
print("5. process_next() devolvió:", devuelto)
print("estado:", p.lista, "| pending():", p.pending())


5. process_next() devolvió: ('S01', 'temperature', 20)
estado: [('S01', 'temperature', 20)] | pending(): 0


In [18]:
#Prueba 6 procesar varios registros
p = DataProcessor()
p.add(A)
p.add(C)

p.process_next()
p.process_next()

print("6. estado tras procesar dos datos distintos:", p.lista)
print("pending():", p.pending())

6. estado tras procesar dos datos distintos: [('S01', 'temperature', 20), ('S01', 'humidity', 60)]
pending(): 0


In [19]:
#Prueba 7 actualizar una variable existente
p = DataProcessor()
p.add(A)
p.add(B)

p.process_next()

print("7. tras procesar A:", p.lista)
p.process_next()

print("tras procesar B (mismo sensor/variable):", p.lista)
print("el estado NO crece, solo cambia el valor:", len(p.lista), "dato(s)")

7. tras procesar A: [('S01', 'temperature', 20)]
tras procesar B (mismo sensor/variable): [('S01', 'temperature', 25)]
el estado NO crece, solo cambia el valor: 1 dato(s)


In [42]:
#Prueba 8 
p = DataProcessor()
p.add(A)
p.add(C)

p.process_next() 
p.process_next()

print("8. current_value('S01','temperature'):", p.current_value("S01", "temperature"))
print("current_value('S01','humidity'):", p.current_value("S01", "humidity"))

8. current_value('S01','temperature'): 20
current_value('S01','humidity'): 60


In [26]:
#Prueba 9 realizar un undo()
p = DataProcessor()
p.add(A)
p.add(B)

p.process_next() 
p.process_next()

print("9. antes del undo:", p.lista)
p.undo()
print("después del undo:", p.lista)



9. antes del undo: [('S01', 'temperature', 25)]
después del undo: [('S01', 'temperature', 20)]


In [27]:
#Prueba 10 varios undo() consecutivos (ejemplo 20 -> 25 -> 30)
p = DataProcessor()
for v in (20, 25, 30):
    p.add(("S01", "temperature", v))
    p.process_next()
print("10. valor inicial:", p.current_value("S01", "temperature"))
p.undo()
print("    tras undo():", p.current_value("S01", "temperature"))
p.undo()
print("    tras undo():", p.current_value("S01", "temperature"))

10. valor inicial: 30
    tras undo(): 25
    tras undo(): 20


In [29]:
#Prueba 11 procesar cuando la queue está vacía -> Empty
p = DataProcessor()
print("11. process_next() con la cola vacía:")
p.process_next()

11. process_next() con la cola vacía:


Empty: La cola de registros pendientes está vacía

In [30]:
#Prueba 12 undo() cuando el historial está vacío -> Empty
p = DataProcessor()
print("12. undo() sin cambios en el historial:")
p.undo()


12. undo() sin cambios en el historial:


Empty: No hay cambios en el historial para deshacer.

In [38]:
#Prueba 13 deshacer la creación de un dato que antes no existía
p = DataProcessor()
p.add(("S01", "temperature", 23.5))
p.process_next()
print("13. el dato existe:", p.current_value("S01", "temperature"))
p.undo()
print("después del undo el estado es:", p.lista)
print("y la consulta vuelve a fallar:")
p.current_value("S01", "temperature")

13. el dato existe: 23.5
después del undo el estado es: []
y la consulta vuelve a fallar:


KeyError: ('S01', 'temperature')

In [33]:
#Prueba 14
p = DataProcessor()
for v in (10, 20, 30, 40):
    p.add(("S07", "presion", v))
valores = []
while p.pending() > 0:
    p.process_next()
    valores.append(p.current_value("S07", "presion"))
print("14. secuencia de valores:", valores)
print("datos distintos en el estado:", len(p.lista))
print("tamaño del historial:", len(p.stack))

14. secuencia de valores: [10, 20, 30, 40]
datos distintos en el estado: 1
tamaño del historial: 4


In [39]:
#Prueba 15 agregar un registro con formato incorrecto -> ValueError
# Prueba 15 corregida:
p = DataProcessor()
print('15. add con formato incorrecto:')
p.add(('S01', 'temperature'))  # Tupla de 2 elementos (falla len != 3)
p.add(('S01', 'temperature', 'no_es_numero'))  # Valor no numérico


print('la cola quedó intacta, pending():', p.pending())

15. add con formato incorrecto:


ValueError: El registro debe ser una tupla de tres componentes: (sensor, variable, value).

In [40]:
# Prueba 16 verificar el funcionamiento de redo()
p = DataProcessor()
p.add(("S03", "voltage", 12.5))
p.process_next()

print("16. estado inicial tras procesar:", p.lista)
p.undo()
print("estado tras el undo():", p.lista)

p.redo()
print("estado tras el redo():", p.lista)
print("el valor actual recuperado es:", p.current_value("S03", "voltage"))


16. estado inicial tras procesar: [('S03', 'voltage', 12.5)]
estado tras el undo(): []
estado tras el redo(): [('S03', 'voltage', 12.5)]
el valor actual recuperado es: 12.5


In [41]:
# Prueba 17 redo() sin historial para rehacer -> Empty
p = DataProcessor()
print("17. intentando hacer redo() con el redo_stack vacío:")

p.redo()

17. intentando hacer redo() con el redo_stack vacío:


Empty: No hay cambios para rehacer (redo).

In [36]:
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
print("pendientes:", p.pending())

p.process_next()
print("tras procesar A:", p.lista)
p.process_next()
print("tras procesar B:", p.lista)
p.process_next()
print("tras procesar C:", p.lista)

p.undo()
print("tras undo (quita C):", p.lista)
p.undo()
print("tras undo (quita B):", p.lista)
p.undo()
print("tras undo (quita A):", p.lista)

pendientes: 3
tras procesar A: [('S01', 'temperature', 20)]
tras procesar B: [('S01', 'temperature', 25)]
tras procesar C: [('S01', 'temperature', 25), ('S01', 'humidity', 60)]
tras undo (quita C): [('S01', 'temperature', 25)]
tras undo (quita B): [('S01', 'temperature', 20)]
tras undo (quita A): []


##### **redo()**

Se utilizó un otro ArrayStack (self.redo_stack), porque funciona similar al undo() el cual recorre el historial en orden LIFO, el útlimo que entra es el primero que sale, y redo() debe recorrer los cambios deshechos en el orden inverso a como se deshicieron, que también es LIFO, por ejemplo, si escribí A, B, aplico undo() dos veces (primero se borra B y luego A) y quiero aplicar redo() primero se pondría a A y luego a B, por eso se recorren en orden inverso. Por esta razón se aplico una pila en vez de usar una cola. 

- undo(), antes de revertir, toma de la lista el registro que se había aplicado y lo empuja a la pila del redo(). Se guarda el registro original y no el elemento del historial porque al volver a aplicarlo la posición y el valor anterior pueden ser distintos. 
- redo(), saca el registro de la pila de rehacer, lo aplica igual que process_next() y empuja el cambio al historial. De esta manera un cambio rehechos se puede volver a deshacer.


In [6]:
p = DataProcessor()
for v in (20, 25, 30):
    p.add(("S01", "temperature", v))
    p.process_next()

print("valor actual:", p.current_value("S01", "temperature"))
p.undo()
print("después de undo():", p.current_value("S01", "temperature"))
p.redo()
print("después de redo():", p.current_value("S01", "temperature"))

p.undo()
p.undo()
print("\ntras dos undo():", p.current_value("S01", "temperature"))

p.redo()
p.redo()
print("tras dos redo():", p.current_value("S01", "temperature"))

valor actual: 30
después de undo(): 25
después de redo(): 30

tras dos undo(): 20
tras dos redo(): 30


In [8]:
# redo() sin nada que rehacer -> Empty
p = DataProcessor()
print("redo() sobre un procesador vacío:")
p.redo

# un cambio nuevo invalida la rama de rehacer
p.add(("S01", "temperature", 20))
p.process_next()
p.add(("S01", "temperature", 25))
p.process_next()
p.undo()
p.add(("S01", "temperature", 99))
p.process_next()
print("\nvalor actual:", p.current_value("S01", "temperature"))
print("redo() después de un cambio nuevo:")
p.redo()

redo() sobre un procesador vacío:

valor actual: 99
redo() después de un cambio nuevo:


Empty: No hay cambios para rehacer (redo).

#### Decisiones de diseño
- El estado es una lista de tuplas, sin diccionarios ya que es lo que pide el enunciado, por esto cada dato se guarda como el registro completo, la tupla con los tres elementos y el (sensor, variable) se localiza recorriendo la lista.
- En el método process_next() están el CREATE y el UPTADE, que sirven más adelante para el undo(), ya que si es CREATE, se elimina el dato puesto que no había nada antes y si es UPTADE, se reescribe el registro anterior. 
- Los datos nuevos se agregan al final de la lista y las actualizaciones no mueven a nadie de su posición, como los undo() ocurren en orden LIFO, al deshacer un CREATE por ejemplo, siempre será el que está ubicado al final de la lista, así que eliminarlo no desplaza a ningpun otro registro.
- El historial guarda el cambio, no una copia del estado, se guarda (tipo (CREATE o UPTADE), posicion, anterior (valor antiguo)) esto hace que ocupe espacio constante y que undo() tenga complejidad O(1) porque la posición ya viene guardada.
- add() revisa el registro y lanza ValueError de inmediato si no cumple con el formatp, ya que es preferible rechazar un dato con el formato incorrecto cuando llego y no cuando se esté procesando porque puede ser complicado después.

##### Complejidad computacional

- add() - O(1) -> en la validación todas las operaciones son de tiempo constante. El casos costoso computacionalmente, sería cuando el arreglo se llena y tendría que hacer resize. 
- process_next() - O(n) -> dequeue, push y la tupla del historial son O(1), pero al recorrer la lista para ver si los (sensor, variable) ya existen, entonces en el peor de los casos debería recorrer todas las entradas por lo que la complejidad sería O(n). 
- undo() - O(1) -> el pop del ArrayStack es O(1). Si el cambio fue 'UPDATE', restaurar es una asignación por índice, O(1). Si fue 'CREATE', se usa list.pop(posicion), por el orden LIFO esa posición siempre es la última de la lista, así que no hay desplazamiento y la operación es O(1). 
- pending() - O(1)
- current_value — O(n) -> es el recorrido lineal de la lista.